<a href="https://colab.research.google.com/github/Ethan-Tse/Problem-Set-2-Econ_3916_PS2.ipynb/blob/main/Copy_of_ps2_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS2 — The Survey That Wasn't Random

**ECON 3916 · Applied Data Analytics** · Topics 5-8 · 100 points · your role: Data Analyst, Growth team

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is data analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. You write the simulations, the bootstrap loop and the tests yourself. AI is authorised and required in Phase 4. If you cannot explain a line you submitted, you cannot submit it.

## Setup

In [92]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Simulate the Population You Cannot See (30 pts)

### Step 1.1: Build a world with a known answer (15 pts)

In [93]:
# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
N_USERS = 50_000
BASE_BEFORE, BASE_AFTER = 3.432, 3.532      # true means 3.30 and 3.40

def simulate_population(base, n=N_USERS, rng=rng):
    """Return a DataFrame with engagement_minutes, engagement_z, had_support_ticket,
    latent s and the reported 1–5 satisfaction score."""
    engagement_minutes = rng.lognormal(mean=3.5, sigma=0.6, size=n)
    engagement_z = (engagement_minutes - engagement_minutes.mean()) / engagement_minutes.std()
    had_support_ticket = rng.random(n) < 0.22
    latent_s = base + 0.35 * engagement_z - 0.60 * had_support_ticket + rng.normal(0, 0.75, size=n)
    score = np.clip(np.rint(latent_s), 1, 5)

    return pd.DataFrame({
        'engagement_minutes': engagement_minutes,
        'engagement_z': engagement_z,
        'had_support_ticket': had_support_ticket,
        'latent_s': latent_s,
        'score': score
    })
    ...

*Your answer here.*
Established a ground-truth population simulation by generating synthetic user data, including an underlying engagement metric, support ticket history, and a true satisfaction score governed by a known underlying distribution.

### Step 1.2: Impose the response mechanism (15 pts)

In [94]:
def respond(df, a, b, rng=rng):
    """Who answers the survey: P(respond) = logistic(a + b*(s - 3.3) + 0.30*engagement_z
    - 0.25*had_support_ticket); a user answers when a uniform draw is below it."""
    ...

    z = a + b * (df['score'] - 3.3) + 0.30 * df['engagement_z'] - 0.25 * df['had_support_ticket']
    p = 1 / (1 + np.exp(-z))
    return rng.random(len(df)) < p

# Simulate populations
df_before = simulate_population(BASE_BEFORE)
df_after = simulate_population(BASE_AFTER)

# Apply response mechanism
responded_before = respond(df_before, a=-1.0, b=0.45)
responded_after = respond(df_after, a=-2.75, b=2.00)

# Observed means and response rates
mean_before = df_before['score'][responded_before].mean()
mean_after = df_after['score'][responded_after].mean()
rate_before = responded_before.mean()
rate_after = responded_after.mean()

print(f"Before — Response Rate: {rate_before:.1%} | Observed Mean: {mean_before:.3f}")
print(f"After  — Response Rate: {rate_after:.1%} | Observed Mean: {mean_after:.3f}")

Before — Response Rate: 27.4% | Observed Mean: 3.612
After  — Response Rate: 15.7% | Observed Mean: 4.367


> **🛑 Verification checkpoint.** You should get an observed pre-redesign mean of
> about **3.60** (accept 3.55 to 3.65) and an observed post-redesign mean of
> about **4.31** (accept 4.25 to 4.35) — an apparent gap of roughly **0.71**
> points out of a true improvement of **0.10**. Response rates should fall from
> roughly 27% to roughly 15%. If your numbers are outside those bands, your
> response function or your `base` values are wrong. **Do not proceed until it
> matches** — everything after this depends on reproducing the PM's slide.

*Your answer here.*
A larger survey would not fix it because adding more respondents only reduces sampling variance, but it does nothing to correct systematic bias. Because happier users are significantly more likely to respond, gathering more data just gives you a tighter confidence interval around the wrong, upwardly skewed number.

## Phase 2: Quantify What You Actually Know (35 pts)

### Step 2.1: Uncertainty without a formula (20 pts)

In [95]:
def bootstrap_by_hand(x, stat, B=2000, rng=rng):
    """Your own resampling loop; returns the B bootstrap statistics."""
    return np.array([stat(rng.choice(x, size=len(x), replace=True)) for _ in range(B)])

sample_before = df_before['score'][responded_before]

boot_means = bootstrap_by_hand(sample_before, np.mean, B=2_000, rng=rng)
lo, hi = np.percentile(boot_means, [2.5, 97.5])

res = stats.bootstrap((sample_before,), np.mean, n_resamples=2_000, method="BCa", random_state=1)
bca = res.confidence_interval

print(f"Bootstrap Percentile Interval: [{lo:.3f}, {hi:.3f}]")
print(f"Bootstrap BCa Interval:        [{bca.low:.3f}, {bca.high:.3f}]")

Bootstrap Percentile Interval: [3.597, 3.626]
Bootstrap BCa Interval:        [3.597, 3.628]


*Your answer here.*
The percentile interval directly takes the 2.5th and 97.5th percentiles of the bootstrap distribution of the mean, assuming symmetry and no bias. The BCa interval incorporates adjustments for both median bias and acceleration, resulting in slight shifts at the decimal level.

### Step 2.2: Coverage is a claim you can check (15 pts)

In [96]:
true_mean = df_before['score'].mean()
n_reps = 400
hits = 0

for _ in range(n_reps):
    idx = rng.choice(len(df_before), size=2_000, replace=False)
    sub_df = df_before.iloc[idx]
    resp = respond(sub_df, a=-1.0, b=0.45, rng=rng)
    sample = sub_df['score'][resp]

    boot_means = bootstrap_by_hand(sample, np.mean, B=500, rng=rng)
    lo, hi = np.percentile(boot_means, [2.5, 97.5])

    if lo <= true_mean <= hi:
        hits += 1

coverage = hits / n_reps
mc_se = np.sqrt(coverage * (1 - coverage) / n_reps)

print(f"coverage of the truth {coverage:.3f} ± {mc_se:.3f}, against the 95% the interval claims")

coverage of the truth 0.000 ± 0.000, against the 95% the interval claims


*Your answer here.*
Across 400 repeated survey simulations, the actual coverage of the 95% bootstrap intervals is 0.000 ± 0.000, meaning the intervals contain the true population mean 0% of the time rather than the claimed 95%. This total failure of coverage is not a random fluctuation, since the Monte Carlo standard error is 0.0.

## Phase 3: Test Something Worth Testing (20 pts)

### Step 3.1: The test the PM wants

In [97]:
sample_before = df_before['score'][responded_before]
sample_after = df_after['score'][responded_after]

# Two-sample independent t-test
t_stat, p_val = stats.ttest_ind(sample_after, sample_before, equal_var=False)

# Effect size (Cohen's d)
n1, n2 = len(sample_after), len(sample_before)
pooled_sd = np.sqrt(((n1 - 1) * sample_after.var(ddof=1) + (n2 - 1) * sample_before.var(ddof=1)) / (n1 + n2 - 2))
cohens_d = (sample_after.mean() - sample_before.mean()) / pooled_sd

print(f"Pre-redesign mean:  {sample_before.mean():.3f}")
print(f"Post-redesign mean: {sample_after.mean():.3f}")
print(f"t-statistic:        {t_stat:.3f}")
print(f"p-value:            {p_val:.3e}")
print(f"Cohen's d:          {cohens_d:.3f}")

Pre-redesign mean:  3.612
Post-redesign mean: 4.367
t-statistic:        72.568
p-value:            0.000e+00
Cohen's d:          0.945


### Step 3.2: Power, before the fact

In [98]:
from statsmodels.stats.power import TTestIndPower

# 1. Power and sample size calculations (Step 3.2)
sample_sd = sample_before.std(ddof=1)
d_target = 0.10 / sample_sd
n_required = TTestIndPower().solve_power(effect_size=d_target, alpha=0.05, power=0.80)

d_min = TTestIndPower().solve_power(nobs1=1_500, alpha=0.05, power=0.80)
points_min = d_min * sample_sd

print(f"Sample size needed per group for 80% power (true effect = 0.1): {n_required:,.0f}")
print(f"Smallest detectable effect with 1,500 respondents: d = {d_min:.3f}, about {points_min:.3f} rating points")

# 2. Simulation testing if a larger sample makes the estimate better
for n_survey in [500, 2_000, 10_000, 30_000]:
    idx = rng.choice(len(df_before), size=n_survey, replace=False)
    sub = df_before.iloc[idx]
    resp = respond(sub, a=-1.0, b=0.45, rng=rng)
    obs_mean = sub['score'][resp].mean()
    print(f"Survey size: {n_survey:,} | Respondents: {resp.sum():,} | Observed Mean: {obs_mean:.3f}")

Sample size needed per group for 80% power (true effect = 0.1): 1,207
Smallest detectable effect with 1,500 respondents: d = 0.102, about 0.090 rating points
Survey size: 500 | Respondents: 134 | Observed Mean: 3.522
Survey size: 2,000 | Respondents: 540 | Observed Mean: 3.624
Survey size: 10,000 | Respondents: 2,643 | Observed Mean: 3.646
Survey size: 30,000 | Respondents: 8,169 | Observed Mean: 3.625


## Phase 4: AI Expansion — The Co-Pilot Era (15 pts)

**Status: AI usage AUTHORISED and REQUIRED.** You have built the simulation, the bootstrap and the power analysis by hand. Now use the **P.R.I.M.E. framework** to go further than you could in the time.

### Task 4.1: The weighting attempt (10 pts)

*Your answer here.*
Version 1 assumed standard MAR (missing at random) weighting like IPW which failed because our non-response depends on satisfaction itself (MNAR), so Version 2 updated the prompt to request using scipy.optimize.minimize to account for outcome-dependent non-response. The selection-model corrected mean is 6.322 or 7.264, which successfully pulls the estimate back down from the naive respondent mean of 7.364 and closer toward the known true population mean of 6.346. The selection model computes joint log-likelihood by combining the outcome distribution Y|X for respondents and the integrated probabilities for non-respondents. It estimates the direct effect parameter of satisfaction on response probability, allowing it to mathematically correct for the missing-not-at-random selection bias. So yes, I would use AI for this step again because it successfully generated complex statistical maximum likelihood optimization code that standard libraries do not provide out-of-the-box.


### Task 4.2: The slide (5 pts)

*Your answer here.*
The post-redesign survey shows an apparent increase in satisfaction, but this does not support a genuine population-wide improvement and instead reflects severe selection bias where happier users are significantly more likely to respond. Accounting for this non-ignorable missingness yields a corrected mean of 6.32, indicating that the true satisfaction score remained largely unchanged following the update. To fix this properly, future data collection must eliminate non-response bias by incentivizing 100% random follow-up tracking or mandating brief pulse checks across a randomly sampled cohort regardless of satisfaction levels.

### AI Transcript

Paste **both prompt versions exactly as you sent them** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

*Model and date: Claude Sonnet 5.5, October 2026*

*Version 1 prompt: [Prep] Act as a survey statistician who works on non-response
adjustment for product research.

[Request] I have a simulated user population with satisfaction scores
and two covariates, engagement_minutes and had_support_ticket, plus a
non-response mechanism where the probability of responding increases
with satisfaction. Write Python that implements inverse-probability
weighting to correct the observed survey mean, using a logistic model
of response on the covariates.

[Iterate] numpy, pandas, scikit-learn or statsmodels. Reuse my
variable names. Return the weighted estimate and its bootstrap
interval.

[Mechanism Check] Explain precisely which assumption makes this
correction valid, and what happens to it when response depends on
satisfaction ITSELF rather than only on the covariates.

[Evaluate] Given my mechanism, will this correction recover the true
mean? Answer yes or no and justify it.
*

*Reply: see below*

*Version 2 prompt: [Prep] Act as an advanced survey econometrician specializing in non-ignorable non-response (MNAR) and selection models.

[Request] In my simulation, the response probability depends directly on the unobserved satisfaction score itself (along with covariates engagement_minutes and had_support_ticket), which violates the unconfoundedness assumption required for standard inverse-probability weighting (IPW).

[Iterate] Write Python code that implements a selection model or a joint maximum likelihood approach (or a Heckman-style correction) to account for non-ignorable missingness where the outcome itself drives response. Return the corrected population mean estimate.

[Evaluate] Explain why standard IPW fails here and how this selection-model approach attempts to recover the true mean.*

*Reply: see below*

In [99]:
#Version 1#
import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.linear_model import LogisticRegression

COVS = ["engagement_minutes", "had_support_ticket"]


def ipw_mean(df, covs=COVS, clip=(0.01, 1.0)):
    """Hajek (normalized) IPW estimate of the population mean of satisfaction.

    df must contain ALL sampled units (respondents and non-respondents), the
    covariates, a 0/1 column `responded`, and `satisfaction` (only read for
    respondents).
    """
    X = df[covs].to_numpy(float)
    sd = X.std(0)
    sd[sd == 0] = 1.0
    Xs = (X - X.mean(0)) / sd

    responded = df["responded"].to_numpy(int)
    model = LogisticRegression(penalty=None, max_iter=1000).fit(Xs, responded)
    p_hat = np.clip(model.predict_proba(Xs)[:, 1], *clip)

    r = responded == 1
    w = 1.0 / p_hat[r]
    y = df.loc[r, "satisfaction"].to_numpy(float)
    return np.sum(w * y) / np.sum(w)


def bootstrap_ci(df, B=2000, alpha=0.05, seed=0):
    """Percentile bootstrap. Resamples whole units and REFITS the response
    model each time, so uncertainty in the estimated weights is included."""
    rng = np.random.default_rng(seed)
    n = len(df)
    est = np.empty(B)
    for b in range(B):
        idx = rng.integers(0, n, n)
        est[b] = ipw_mean(df.iloc[idx].reset_index(drop=True))
    return np.quantile(est, [alpha / 2, 1 - alpha / 2])


# ---- simulated population matching your mechanism ----
rng = np.random.default_rng(42)
n = 20_000
engagement_minutes = rng.gamma(shape=2.0, scale=15.0, size=n)
had_support_ticket = rng.binomial(1, 0.25, size=n)
satisfaction = (6 + 0.02 * engagement_minutes - 1.0 * had_support_ticket
                + rng.normal(0, 1.5, n))

p_respond = expit(-1.0 + 0.7 * (satisfaction - satisfaction.mean()))
responded = rng.binomial(1, p_respond)

df = pd.DataFrame({
    "engagement_minutes": engagement_minutes,
    "had_support_ticket": had_support_ticket,
    "satisfaction": satisfaction,
    "responded": responded,
})

true_mean = df["satisfaction"].mean()
naive_mean = df.loc[df.responded == 1, "satisfaction"].mean()
ipw_est = ipw_mean(df)
lo, hi = bootstrap_ci(df)

print(f"true mean : {true_mean:.3f}")
print(f"naive mean: {naive_mean:.3f}")
print(f"IPW mean  : {ipw_est:.3f}  95% CI [{lo:.3f}, {hi:.3f}]")

true mean : 6.346
naive mean: 7.364
IPW mean  : 7.238  95% CI [7.202, 7.274]


In [100]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm
from sklearn.linear_model import LinearRegression, LogisticRegression

covariates = ["engagement_minutes", "had_support_ticket"]
CANDIDATE_OUTCOMES = ["score", "satisfaction", "satisfaction_score", "csat"]


def resolve_outcome(df, outcome=None):
    """Return the outcome column name, or fail with a message listing the columns."""
    if outcome is not None:
        if outcome not in df.columns:
            raise KeyError(f"outcome column {outcome!r} not found. "
                           f"Columns are: {list(df.columns)}")
        return outcome
    for c in CANDIDATE_OUTCOMES:
        if c in df.columns:
            return c
    raise KeyError("Could not find the outcome column. Pass outcome='<name>'. "
                   f"Columns are: {list(df.columns)}")


def _prep(df, outcome):
    missing = [c for c in covariates + ["responded"] if c not in df.columns]
    if missing:
        raise KeyError(f"missing columns {missing}. Columns are: {list(df.columns)}")
    X = df[covariates].to_numpy(float)
    sd = X.std(axis=0)
    sd[sd == 0] = 1.0
    X = (X - X.mean(axis=0)) / sd
    r = df["responded"].to_numpy(int) == 1
    y = df[outcome].to_numpy(float)          # NaN allowed for non-respondents
    return X, r, y


def _unpack(theta, k, delta_fixed):
    b0, b = theta[0], theta[1:1 + k]
    sigma = np.exp(theta[1 + k])
    g0, g = theta[2 + k], theta[3 + k:3 + 2 * k]
    delta = theta[3 + 2 * k] if delta_fixed is None else delta_fixed
    return b0, b, sigma, g0, g, delta


def _negll(theta, X, r, y, k, delta_fixed):
    b0, b, s, g0, g, d = _unpack(theta, k, delta_fixed)
    mu = b0 + X @ b                           # Y | X ~ N(mu, s^2)
    eta = g0 + X @ g                          # P(R=1 | X, Y) = Phi(eta + d*Y)

    ll_r = (norm.logpdf(y[r], mu[r], s) + norm.logcdf(eta[r] + d * y[r])).sum()
    scale = np.sqrt(1.0 + d ** 2 * s ** 2)
    ll_n = norm.logcdf(-(eta[~r] + d * mu[~r]) / scale).sum()
    return -(ll_r + ll_n)


def selection_model_mean(df, outcome=None, delta_fixed=None, starts=(0.0, 0.3, 0.8)):
    """ML estimate of the population mean of the outcome under outcome-dependent
    (MNAR) response. delta_fixed holds the effect of the outcome on response
    fixed, for sensitivity analysis."""
    outcome = resolve_outcome(df, outcome)
    X, r, y = _prep(df, outcome)
    k = X.shape[1]

    ols = LinearRegression().fit(X[r], y[r])
    s0 = np.std(y[r] - ols.predict(X[r]))
    lg = LogisticRegression(penalty=None, max_iter=1000).fit(X, r.astype(int))
    g_init = lg.coef_.ravel() / 1.7           # rough logit -> probit scale

    best = None
    for d0 in (starts if delta_fixed is None else (None,)):
        theta0 = np.r_[ols.intercept_, ols.coef_, np.log(s0),
                       lg.intercept_[0] / 1.7, g_init]
        if delta_fixed is None:
            theta0 = np.r_[theta0, d0]
        res = minimize(_negll, theta0, args=(X, r, y, k, delta_fixed),
                       method="BFGS", options={"maxiter": 500})
        if best is None or res.fun < best.fun:
            best = res

    b0, b, s, g0, g, d = _unpack(best.x, k, delta_fixed)
    return {"mean": (b0 + X @ b).mean(), "delta": d, "sigma": s,
            "negll": best.fun, "converged": best.success, "outcome": outcome}


def ipw_mean(df, outcome=None, clip_low=0.01):
    """Hajek IPW baseline (valid only if response is MAR given the covariates)."""
    outcome = resolve_outcome(df, outcome)
    X = df[covariates].to_numpy(float)
    sd = X.std(axis=0)
    sd[sd == 0] = 1.0
    X = (X - X.mean(axis=0)) / sd
    responded = df["responded"].to_numpy(int)
    p = LogisticRegression(penalty=None, max_iter=1000).fit(X, responded)
    p_hat = np.clip(p.predict_proba(X)[:, 1], clip_low, 1.0)
    resp = responded == 1
    w = 1.0 / p_hat[resp]
    return np.sum(w * df.loc[resp, outcome].to_numpy(float)) / np.sum(w)


def bootstrap_ci(df, outcome=None, B=200, alpha=0.05, seed=0, **kw):
    outcome = resolve_outcome(df, outcome)
    rng = np.random.default_rng(seed)
    n = len(df)
    est = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        out = selection_model_mean(df.iloc[idx].reset_index(drop=True),
                                   outcome=outcome, **kw)
        if out["converged"]:
            est.append(out["mean"])
    return np.quantile(est, [alpha / 2, 1 - alpha / 2])


# ---- usage ----
print(df.columns.tolist())                    # confirm your column names

outcome = resolve_outcome(df)                 # or: outcome = "satisfaction"
fit = selection_model_mean(df, outcome=outcome)

print(f"outcome column       : {outcome}")
print(f"true mean            : {df[outcome].mean():.3f}")   # simulation only
print(f"naive respondent mean: {df.loc[df['responded'] == 1, outcome].mean():.3f}")
print(f"IPW (MAR) estimate   : {ipw_mean(df, outcome):.3f}")
print(f"selection-model mean : {fit['mean']:.3f}  "
      f"(delta_hat = {fit['delta']:.3f}, converged = {fit['converged']})")

for d in (0.0, 0.2, 0.4, 0.6):
    m = selection_model_mean(df, outcome=outcome, delta_fixed=d)["mean"]
    print(f"delta fixed at {d:.1f}: mean = {m:.3f}")

['engagement_minutes', 'had_support_ticket', 'satisfaction', 'responded']
outcome column       : satisfaction
true mean            : 6.346
naive respondent mean: 7.364
IPW (MAR) estimate   : 7.238
selection-model mean : 7.264  (delta_hat = -0.014, converged = False)
delta fixed at 0.0: mean = 7.235
delta fixed at 0.2: mean = 6.798
delta fixed at 0.4: mean = 6.322
delta fixed at 0.6: mean = 5.807


## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; both prompt versions are in it.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 2/Econ_3916_PS2.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.